# RAG From Scratch — Virtual Liaison Project Data

This notebook implements a **minimal Retrieval-Augmented Generation (RAG) pipeline from scratch**,
mirroring the "RAG-enabled Project-based data retrieval using langchain & pinecone" facility of the
Indegene Virtual Liaison platform (see `../01-rag-architecture-deep-dive.md`).

Everything here runs **fully offline**:
- A tiny synthetic "project documents" corpus stands in for real project status notes.
- **TF-IDF cosine similarity** (scikit-learn) stands in for a real embedding model + vector search.
- A **mock generation step** stuffs retrieved context into a prompt template and prints it, standing
  in for an actual LLM call (no API key required).

Run the cells top to bottom.

In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", 100)
print("Libraries loaded.")

Libraries loaded.


## 1. A tiny synthetic "project documents" corpus

Each document mimics one chunk of project data at Indegene: a status update, a revision note, or a
cost-catalog line — the same kind of source material the real platform would chunk and embed
(see Chapter 1, "Chunking").

In [2]:
documents = [
    {"doc_id": "atlas-status-01", "project": "Project Atlas",
     "text": "Project Atlas Japan localization is on track for a Q3 launch. Translation review "
             "completed on schedule."},
    {"doc_id": "atlas-status-02", "project": "Project Atlas",
     "text": "Project Atlas France localization is delayed two weeks pending legal review of the "
             "updated promotional claims."},
    {"doc_id": "atlas-cost-01", "project": "Project Atlas",
     "text": "Project Atlas cost catalog: SKU-LOC-JP-STD Japanese localization standard package, "
             "estimated 5 business days turnaround."},
    {"doc_id": "orion-status-01", "project": "Project Orion",
     "text": "Project Orion regulatory submission package for the EU region passed compliance "
             "review and moved to final formatting."},
    {"doc_id": "orion-status-02", "project": "Project Orion",
     "text": "Project Orion German localization kicked off this week, targeting a 10 business day "
             "turnaround for the training deck."},
    {"doc_id": "nova-status-01", "project": "Project Nova",
     "text": "Project Nova oncology promotional deck localization into Korean is in the design "
             "review stage, no blockers reported."},
]

corpus_df = pd.DataFrame(documents)
corpus_df

,doc_id,project,text
0,atlas-status-01,Project Atlas,Project Atlas Japan localization is on track for a Q3 launch. Translation review completed on sc...
1,atlas-status-02,Project Atlas,Project Atlas France localization is delayed two weeks pending legal review of the updated promo...
2,atlas-cost-01,Project Atlas,"Project Atlas cost catalog: SKU-LOC-JP-STD Japanese localization standard package, estimated 5 b..."
3,orion-status-01,Project Orion,Project Orion regulatory submission package for the EU region passed compliance review and moved...
4,orion-status-02,Project Orion,"Project Orion German localization kicked off this week, targeting a 10 business day turnaround f..."
5,nova-status-01,Project Nova,"Project Nova oncology promotional deck localization into Korean is in the design review stage, n..."


## 2. Chunking (already-chunked here)

In production, a `RecursiveCharacterTextSplitter`-style splitter (Chapter 1) would split long project
histories into chunks like the ones above. Our synthetic corpus is already chunk-sized, so we skip
straight to embedding/indexing — but note each "document" here is exactly the unit of retrieval:
one coherent status update or catalog line, not an arbitrary character window.

## 3. "Embedding" stand-in: TF-IDF vectorization

A real pipeline would call an embedding model (e.g. `text-embedding-3-small`) to turn each chunk into
a dense vector. Here, TF-IDF gives us a transparent, fully offline stand-in: each document becomes a
sparse vector weighted by term frequency / inverse document frequency, and cosine similarity between
vectors approximates semantic closeness for this toy example.

In [3]:
vectorizer = TfidfVectorizer(stop_words="english")
doc_matrix = vectorizer.fit_transform(corpus_df["text"])

print(f"Vocabulary size: {len(vectorizer.vocabulary_)}")
print(f"Document-term matrix shape: {doc_matrix.shape}")

Vocabulary size: 57
Document-term matrix shape: (6, 57)


## 4. Retrieval: top-K nearest documents to a query

This mirrors `retriever.invoke(query)` from Chapter 1 — embed the query with the *same* vectorizer,
then rank corpus documents by cosine similarity.

In [4]:
def retrieve(query: str, top_k: int = 2):
    query_vec = vectorizer.transform([query])
    sims = cosine_similarity(query_vec, doc_matrix).ravel()
    ranked_idx = np.argsort(sims)[::-1][:top_k]
    results = corpus_df.iloc[ranked_idx].copy()
    results["similarity"] = sims[ranked_idx]
    return results.reset_index(drop=True)


query = "What's the status of the France localization for Project Atlas?"
retrieved = retrieve(query, top_k=2)
retrieved

,doc_id,project,text,similarity
0,atlas-status-02,Project Atlas,Project Atlas France localization is delayed two weeks pending legal review of the updated promo...,0.465090
1,atlas-status-01,Project Atlas,Project Atlas Japan localization is on track for a Q3 launch. Translation review completed on sc...,0.234162


## 5. Generation: stuff retrieved context into a prompt template

This is the `rag_prompt` from Chapter 1, with a **mock LLM** standing in for a real API call — it
doesn't generate free text, it just deterministically renders what a real model would receive so you
can inspect exactly what "grounding in retrieved context" looks like on the wire.

In [5]:
RAG_PROMPT_TEMPLATE = """SYSTEM: You are the Virtual Liaison assistant. Answer only using the
context below. If the answer isn't in the context, say you don't have that information yet.

CONTEXT:
{context}

QUESTION: {question}

ANSWER:"""


def mock_llm_generate(prompt: str) -> str:
    """Stand-in for a real LLM call (e.g. AzureChatOpenAI.invoke).

    A real implementation would do something like:
        from langchain_openai import AzureChatOpenAI
        llm = AzureChatOpenAI(azure_deployment="...")
        return llm.invoke(prompt).content

    Here we just simulate a grounded answer by extracting the most relevant retrieved sentence,
    to prove the context was actually used -- no API key, no network call.
    """
    # naive 'grounded' answer: return the single most similar retrieved line
    return "(mock LLM) Based on the retrieved context: " + prompt.split("CONTEXT:\n")[1].split("\n\n")[0].split("\n")[0]


def rag_answer(question: str, top_k: int = 2) -> str:
    retrieved_docs = retrieve(question, top_k=top_k)
    context = "\n".join(f"- [{r.doc_id}] {r.text}" for r in retrieved_docs.itertuples())
    prompt = RAG_PROMPT_TEMPLATE.format(context=context, question=question)
    print("----- PROMPT SENT TO LLM -----")
    print(prompt)
    print("-------------------------------")
    answer = mock_llm_generate(prompt)
    return answer


answer = rag_answer("What's the status of the France localization for Project Atlas?")
print("\nFINAL ANSWER:", answer)

----- PROMPT SENT TO LLM -----
SYSTEM: You are the Virtual Liaison assistant. Answer only using the
context below. If the answer isn't in the context, say you don't have that information yet.

CONTEXT:
- [atlas-status-02] Project Atlas France localization is delayed two weeks pending legal review of the updated promotional claims.
- [atlas-status-01] Project Atlas Japan localization is on track for a Q3 launch. Translation review completed on schedule.

QUESTION: What's the status of the France localization for Project Atlas?

ANSWER:
-------------------------------

FINAL ANSWER: (mock LLM) Based on the retrieved context: - [atlas-status-02] Project Atlas France localization is delayed two weeks pending legal review of the updated promotional claims.


## 6. Try another query

Notice how retrieval changes based on the question -- a cost question should surface the catalog
chunk, not a status chunk.

In [6]:
answer2 = rag_answer("How much does the Japanese localization cost for Project Atlas?", top_k=2)
print("\nFINAL ANSWER:", answer2)

----- PROMPT SENT TO LLM -----
SYSTEM: You are the Virtual Liaison assistant. Answer only using the
context below. If the answer isn't in the context, say you don't have that information yet.

CONTEXT:
- [atlas-cost-01] Project Atlas cost catalog: SKU-LOC-JP-STD Japanese localization standard package, estimated 5 business days turnaround.
- [atlas-status-01] Project Atlas Japan localization is on track for a Q3 launch. Translation review completed on schedule.

QUESTION: How much does the Japanese localization cost for Project Atlas?

ANSWER:
-------------------------------

FINAL ANSWER: (mock LLM) Based on the retrieved context: - [atlas-cost-01] Project Atlas cost catalog: SKU-LOC-JP-STD Japanese localization standard package, estimated 5 business days turnaround.


## Takeaways

- Retrieval quality entirely determines what the "LLM" (mock or real) can ground its answer in --
  garbage/irrelevant retrieval means a garbage/irrelevant answer, no matter how good the generation
  model is. This is why Chapters 1-3 spend so much time on chunking, embeddings, and hybrid search.
- Swapping the `TfidfVectorizer` here for a real embedding model (Chapter 1) and the in-memory
  `doc_matrix` for a Pinecone index (Chapter 2) is a drop-in replacement -- `retrieve()`'s signature
  doesn't need to change, only its implementation.
- The prompt template + mock generation step shows exactly what "grounding" means mechanically: the
  retrieved text is concatenated into the prompt, and the model is instructed not to answer beyond
  it.